In [ ]:
import kagglehub
path = kagglehub.dataset_download("uom190346a/e-commerce-customer-behavior-dataset")
print("Path to dataset files:", path)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

print("Files available:", os.listdir(path))

csv_files = [file for file in os.listdir(path) if file.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("No CSV file was found in the downloaded dataset folder.")

expected_file = "E-commerce Customer Behavior - Sheet1.csv"
csv_path = os.path.join(path, expected_file) if expected_file in csv_files else os.path.join(path, csv_files[0])

df = pd.read_csv(csv_path)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nFirst five rows:")
display(df.head())

print("\\nNumerical summary:")
display(df.describe(include=[np.number]).T)

print("\\nSummary of all columns:")
display(df.describe(include="all").T)

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
print("Missing values by column:")
display(missing_values.to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())

print("Unique values in each column:")
display(df.nunique().sort_values().to_frame("unique_values"))

In [ ]:
df.columns = df.columns.astype(str).str.strip()
df = df.drop_duplicates().copy()

print("Shape after removing exact duplicate rows:", df.shape)
print("Column names:")
print(df.columns.tolist())

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

if numeric_cols:
    df[numeric_cols].hist(figsize=(14, 10), bins=20, edgecolor="black")
    plt.suptitle("Distributions of Numerical Features", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns were detected.")

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

if categorical_cols:
    n_cols = 2
    n_rows = int(np.ceil(len(categorical_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, categorical_cols):
        counts = df[col].astype("string").fillna("Missing").value_counts().head(15)
        sns.barplot(x=counts.values, y=counts.index, ax=ax)
        ax.set_title(f"Value Counts: {col}")
        ax.set_xlabel("Count")
        ax.set_ylabel(col)

    for ax in axes[len(categorical_cols):]:
        ax.remove()

    plt.tight_layout()
    plt.show()
else:
    print("No categorical columns were detected.")

In [ ]:
if len(numeric_cols) >= 2:
    plt.figure(figsize=(12, 8))
    correlation = df[numeric_cols].corr(numeric_only=True)
    sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numerical columns are needed for a correlation heatmap.")

In [ ]:
target_candidates = [
    "Satisfaction Level", "Satisfaction", "Customer Satisfaction",
    "Satisfaction_Level", "satisfaction_level", "satisfaction"
]

normalised_columns = {str(col).strip().lower().replace("_", " ").replace("-", " "): col for col in df.columns}
target_col = None

for candidate in target_candidates:
    key = candidate.lower().replace("_", " ").replace("-", " ")
    if key in normalised_columns:
        target_col = normalised_columns[key]
        break

if target_col is None:
    possible = [col for col in df.columns if "satisfaction" in str(col).lower()]
    if possible:
        target_col = possible[0]

if target_col is None:
    raise ValueError(
        "Could not identify the satisfaction target column. "
        "Please check df.columns and set target_col to the exact target-column name."
    )

print("Selected target column:", target_col)
print("\\nTarget values (including missing):")
print(df[target_col].value_counts(dropna=False))

In [ ]:
target_counts = df[target_col].astype("string").fillna("Missing").value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=target_counts.index, y=target_counts.values)
plt.title("Customer Satisfaction Class Distribution")
plt.xlabel("Satisfaction Class")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print("Class proportions:")
display((df[target_col].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("percentage"))

In [ ]:
model_df = df.dropna(subset=[target_col]).copy()
X = model_df.drop(columns=[target_col])
y = model_df[target_col].astype(str).str.strip()

X = X.dropna(axis=1, how="all")

id_like_cols = [
    col for col in X.columns
    if str(col).strip().lower().replace("_", " ").replace("-", " ") in
    {"customer id", "customerid", "id", "user id", "userid", "row id"}
]
if id_like_cols:
    print("Dropping identifier column(s):", id_like_cols)
    X = X.drop(columns=id_like_cols)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", sorted(y.unique().tolist()))
display(X.head())

In [ ]:
from sklearn.model_selection import train_test_split

class_counts = y.value_counts()
stratify_target = y if len(class_counts) > 1 and class_counts.min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=stratify_target
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("\\nTraining target distribution:")
display(y_train.value_counts(normalize=True).round(3).to_frame("proportion"))
print("Testing target distribution:")
display(y_test.value_counts(normalize=True).round(3).to_frame("proportion"))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}

print("Models prepared:", list(models.keys()))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

trained_models = {}
results = []

for name, classifier in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    trained_models[name] = pipeline
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (weighted)": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (weighted)", ascending=False).reset_index(drop=True)
display(results_df.round(4))

In [ ]:
plot_df = results_df.set_index("Model")[["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"]]
plot_df.plot(kind="bar", figsize=(12, 6), ylim=(0, 1))
plt.title("Classification Model Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model based on weighted F1-score:", best_model_name)
print("\\nClassification report:")
print(classification_report(y_test, best_predictions, zero_division=0))

In [ ]:
labels = sorted(y.unique().tolist())
cm = confusion_matrix(y_test, best_predictions, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

permutation_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="f1_weighted",
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": permutation_result.importances_mean,
    "Std": permutation_result.importances_std
}).sort_values("Importance", ascending=False)

display(importance_df.head(15).round(4))

top_features = importance_df.head(15).sort_values("Importance")
plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"].astype(str), top_features["Importance"], xerr=top_features["Std"])
plt.title("Top Features by Permutation Importance")
plt.xlabel("Decrease in weighted F1-score when shuffled")
plt.tight_layout()
plt.show()

In [ ]:
sample_customer = X_test.iloc[[0]]
actual_label = y_test.iloc[0]
predicted_label = best_model.predict(sample_customer)[0]

print("Customer input:")
display(sample_customer)
print("Actual satisfaction:", actual_label)
print("Predicted satisfaction:", predicted_label)

if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(sample_customer)[0]
    probability_df = pd.DataFrame({
        "Satisfaction class": best_model.named_steps["classifier"].classes_,
        "Predicted probability": probabilities
    }).sort_values("Predicted probability", ascending=False)
    display(probability_df.round(4))